In [16]:
# interactive trajectory intercept minimizer
import importlib

import matplotlib.pyplot as plt
from IPython.display import clear_output
from ipywidgets import interact, IntSlider, FloatSlider, Layout, interactive, Button, HTML, Checkbox
import numpy as np
from orbitengine.body import Body
import orbitengine.engine as oe
import astropy.units as u
from scipy.spatial.transform import Rotation as R
from scipy.optimize import minimize
import time

width = '500px'
target_angle_slider = FloatSlider(min=-.5, max=.1, value=0.01,step=0.01, layout=Layout(width=width), description='Target angle')
thrust_angle_slider = FloatSlider(min=0, max=np.pi, value=1.63,step=0.01, layout=Layout(width=width), description='Thrust angle')
t_flight_slider = IntSlider(min=10, max=500,value=189, layout=Layout(width=width), description='Flight time')
velocity_slider = FloatSlider(min=0, max=5, step=0.01, value=2.12, layout=Layout(width=width), description='Velocity')
mass_slider = FloatSlider(min=10, max=10000, step=1, value=5000, layout=Layout(width=width), description='Mass')
radius_slider = FloatSlider(min=0.001, max=2, step=.001, value=1, layout=Layout(width=width), description='Radius')
style_html = HTML('<style>' + open('notebook_darkmode.css').read() + '</style>')
# Create the checkbox
drag_checkbox = Checkbox(value=False, description='Enable Drag')
temp_checkbox = Checkbox(value=False, description='Enable Temperature')


def target_dist(x, thrust_angle, state_init, state_target, acc_params=None, hist=None):
    velocity, t_flight = x
    rot = R.from_euler('z', thrust_angle)
    v = np.array([velocity, 0, 0])*u.km/u.s

    state_maneuver = Body.State(state_init.position, 
                     rot.apply(v)*u.km/u.s, 
                     state_init.mass)

    sm = state_maneuver.propagate(oe.EARTH_K, t_flight*u.s, acc_params=acc_params)
    st = state_target.propagate(oe.EARTH_K, t_flight*u.s)

    dp = np.linalg.norm(st.position - sm.position).value
    if hist is not None:
        hist.append((velocity, t_flight, dp))
    return dp


def f(target_angle, 
      thrust_angle, 
      t_flight, 
      velocity, 
      mass, 
      radius,
      enable_drag, 
      enable_temp):

    # Define a function that creates a plot
    state_init = Body.State(
    np.array([0,oe.EARTH_RADIUS_KM, 0])*u.km, 
    oe.V_ZERO, 
    mass*u.kg, 
    parent_axis_angle=oe.EARTH_AXIS_ANGLE_Z)

    rot_target = R.from_euler('z', target_angle)
    state_target = Body.State(rot_target.apply(state_init.position)*u.km, 
                              oe.V_ZERO, 
                              mass*u.kg,
                              parent_axis_angle=oe.EARTH_AXIS_ANGLE_Z)

    x0 = [velocity, t_flight]

    print(x0, target_dist(x0, thrust_angle, state_init, state_target))
    ts_start = time.time()
    acc_params = oe.AccParams(
        axial_cross_section=np.pi*(radius*u.m)**2,
        atmosphere_axial_drag_coefficient=0.3,
        enable_drag=enable_drag)

    bounds = [(0, 100), (1, 5000)]
    hist = []
    res = minimize(target_dist, 
                   x0, 
                   bounds=bounds,
                   args=(thrust_angle, 
                         state_init, 
                         state_target, 
                         acc_params, hist))
    ts_stop = time.time()
    print(f"Optimization time: {ts_stop - ts_start:.3f} s")

    # original maneuver
    rot = R.from_euler('z', thrust_angle)
    v = np.array([velocity, 0, 0])*u.km/u.s
    state_guess_maneuver = Body.State(state_init.position, rot.apply(v)*u.km/u.s, state_init.mass)

    ts1 = np.linspace(0, t_flight*u.s, 100)
    sgm = state_guess_maneuver.propagate(oe.EARTH_K, ts1, acc_params=acc_params)
    st_guess = state_target.propagate(oe.EARTH_K, t_flight*u.s)

    plt.style.use('dark_background')
    fig, axs = plt.subplots(1, 1, figsize=(5, 5))

    # plot original inital guess maneuver trajectory
    axs.plot([s.position[0].value for s in sgm], [s.position[1].value for s in sgm], color='b')
    axs.plot(sgm[-1].position[0].value, sgm[-1].position[1].value, 'bo')
    axs.plot(st_guess.position[0].value, st_guess.position[1].value, 'ro', 
              markerfacecolor='none', markeredgecolor='r', markersize=10)

    # optimized maneuver
    velocity, t_flight = res.x
    print(f"Optimized t_flight: {t_flight*u.s:.2f}")
    print(f"Optimized velocity: {velocity*u.km/u.s:.2f}")
    rot = R.from_euler('z', thrust_angle)
    v = np.array([velocity, 0, 0])*u.km/u.s
    state_optimized_maneuver = Body.State(state_init.position, rot.apply(v)*u.km/u.s, state_init.mass, T=oe.TEMP_EARTH)

    #plot optimized trajectory
    ts2 = np.linspace(0, t_flight*u.s, 100)
#    acc_params.enable_temp = enable_temp
    sm2 = state_optimized_maneuver.propagate(oe.EARTH_K, ts2, acc_params=acc_params)
    axs.plot([s.position[0].value for s in sm2], [s.position[1].value for s in sm2],color='g')
    axs.plot(sm2[-1].position[0].value, sm2[-1].position[1].value, 'go')

    st_final = state_target.propagate(oe.EARTH_K, t_flight*u.s)
    axs.plot(st_final.position[0].value, st_final.position[1].value, 'ro')

    #print plot distance
    # print(f"x guess: {x0}")
    # s_guess = state_guess_maneuver.propagate(oe.EARTH_K, x0[1]*u.s)
    # print(s_guess.position.value, st_final)
    # dp = np.linalg.norm(s_guess.position - st_final.position)
    # print(f"plot dp guess: {dp}")

    # print(f"x optimized: {res.x}")
    # dp = np.linalg.norm(sm2[-1].position - st_final.position)
    # print(f"plot dp optimized: {dp}")

    axs.add_artist(plt.Line2D((0, st_final.position[0].value), (0, st_final.position[1].value), color='r', linestyle='dotted'))
    axs.plot(state_init.position[0].value, state_init.position[1].value, 'go')

    # dr = np.linalg.norm(st_final.position - sm2[-1].position)
    # print(f"Distance to target: {dr:.3f}")
 
    circle = plt.Circle((0, 0), oe.EARTH_RADIUS_KM, color='b', fill=False, linestyle='dotted')
    axs.add_artist(circle)
    axs.add_artist(plt.Line2D((0, state_init.position[0].value), (0, state_init.position[1].value), color='g', linestyle='dotted'))
    axs.set_aspect('equal', adjustable='box')
    plt.show()

    hist = np.array(hist)
    err = np.array(hist)[:,2]
    plt.plot(hist[:,0],label='velocity')
    plt.plot(hist[:,1], label='t_flight')
    plt.plot(hist[:,2], label='error')
    plt.legend()
    plt.show()

    # plt.plot(ts2, [s.temperature.value for s in sm2])
    # plt.show()



# Define a function to be run when the button is clicked
def on_button_clicked(b):

    clear_output(wait=True)
    display(style_html,
            drag_checkbox,
            temp_checkbox,
            target_angle_slider,
            thrust_angle_slider, 
            mass_slider, 
            radius_slider,
            t_flight_slider, 
            velocity_slider, 
            button)
    

    # Get the current values of the sliders
    target_angle = target_angle_slider.value
    thrust_angle = thrust_angle_slider.value
    t_flight = t_flight_slider.value
    velocity = velocity_slider.value
    mass = mass_slider.value
    enable_drag = drag_checkbox.value
    enable_temp = temp_checkbox.value
    radius = radius_slider.value
    # Call your function with these values    
    f(target_angle, thrust_angle, t_flight, velocity, mass, radius, enable_drag, enable_temp)

# Set this function to be called when the button is clicked
button = Button(description="Run")

button.on_click(on_button_clicked)

# Display the button
display(style_html,
    drag_checkbox,
    temp_checkbox,
    target_angle_slider,
    thrust_angle_slider, 
    mass_slider, 
    radius_slider,
    t_flight_slider, 
    velocity_slider, 
    button)

HTML(value='<style>.cell-output-ipywidget-background {\n   background-color: transparent !important;\n}\n.jp-O…

Checkbox(value=False, description='Enable Drag')

Checkbox(value=False, description='Enable Temperature')

FloatSlider(value=0.01, description='Target angle', layout=Layout(width='500px'), max=0.1, min=-0.5, step=0.01…

FloatSlider(value=1.63, description='Thrust angle', layout=Layout(width='500px'), max=3.141592653589793, step=…

FloatSlider(value=5000.0, description='Mass', layout=Layout(width='500px'), max=10000.0, min=10.0, step=1.0)

FloatSlider(value=1.0, description='Radius', layout=Layout(width='500px'), max=2.0, min=0.001, step=0.001)

IntSlider(value=189, description='Flight time', layout=Layout(width='500px'), max=500, min=10)

FloatSlider(value=2.12, description='Velocity', layout=Layout(width='500px'), max=5.0, step=0.01)

Button(description='Run', style=ButtonStyle())